<a href="https://colab.research.google.com/github/giusedmb/Embedded-System---Number-Recogniser-with-Lattice-Board/blob/main/hls4ml_for_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/nghielme/hls4ml.git@bambu-backend
!pip install tensorflow-model-optimization

  Cloning https://github.com/nghielme/hls4ml.git (to revision bambu-backend) to /tmp/pip-req-build-moi6q2pb
  Running command git clone --filter=blob:none --quiet https://github.com/nghielme/hls4ml.git /tmp/pip-req-build-moi6q2pb
  Running command git checkout -b bambu-backend --track origin/bambu-backend
  Switched to a new branch 'bambu-backend'
  Branch 'bambu-backend' set up to track remote branch 'bambu-backend' from 'origin'.
  Resolved https://github.com/nghielme/hls4ml.git to commit df1df41b381f0cd9a67bd2f74cf9658d88e50d82
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hls4ml: filename=hls4ml-0.5.0.dev607+gdf1df41b-py3-none-any.whl size=3704748 sha256=fcd6614a1fc9b2b8447431863b12ce537674e03b246a5bbc077c7b91b13f1cfc
  Stored in directory: /tmp/pip-ephem-wheel-cache-71n149bp/wheels/2e/35/22/a6f8e97724ff1fc5a341b93

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LeNet-5 → Bambu HLS — Pipeline completa con testbench integrato
# Target: Lattice ECP5-85 @ 25 MHz (clock_period=40 ns)
# ═══════════════════════════════════════════════════════════════════

import os
import shutil
import numpy as np
import tensorflow as tf
import hls4ml
from tensorflow.keras.models import load_model
from tensorflow.keras.datasets import mnist
from google.colab import files

# ─────────────────────────────────────────────────────────────────
# CONFIGURAZIONE
# ─────────────────────────────────────────────────────────────────

MODEL_PATH      = "lenet_mnist_model.h5"
OUTPUT_DIR      = "12_5"
BUNDLE_ZIP      = "12_5"

PRECISION = "ap_fixed<12,5>"
CLOCK_PERIOD_NS = 40          # 25 MHz
STRATEGY        = "Resource"  # minimizza area (condivide i moltiplicatori)

# ─────────────────────────────────────────────────────────────────
# MAC e RF
#
# Regola fondamentale (assertion in nnet_conv2d_resource.h):
#   RF_conv <= filt_h * filt_w * n_chan_IN   (NON moltiplicato per n_filtri)
#
# Per le Dense: RF deve essere un divisore esatto di n_mac.
# ─────────────────────────────────────────────────────────────────

MAC = {
    "conv1":  5 * 5 * 1 * 6,    # 150
    "conv2":  5 * 5 * 6 * 16,   # 2400
    "fc1":    400 * 120,         # 48000
    "fc2":    120 * 84,          # 10080
    "output": 84 * 10,           # 840
}

# Limite conv = filt_h * filt_w * n_chan_IN (massimo consentito dall'assertion)
# Limite dense = il più grande divisore <= 128

def best_divisor(n, limit):
    """Restituisce il divisore più grande di n che sia <= limit."""
    for d in range(limit, 0, -1):
        if n % d == 0:
            return d
    return 1  # fallback di sicurezza (1 divide sempre)

RF = {
    "conv1":  best_divisor(MAC["conv1"],  5 * 5 * 1),   # max=25,  risultato=25
    "conv2":  best_divisor(MAC["conv2"],  5 * 5 * 6),   # max=150, risultato=150
    "fc1":    best_divisor(MAC["fc1"],    128),          # risultato=128
    "fc2":    best_divisor(MAC["fc2"],    84),           # risultato=84
    "output": best_divisor(MAC["output"], 84),           # risultato=84
    "pool":   1,
    "default": 8,
}

# ─────────────────────────────────────────────────────────────────
# BUILD MODELLO (API Funzionale — richiesta da hls4ml)
# ─────────────────────────────────────────────────────────────────

def build_model():
    original    = load_model(MODEL_PATH, compile=False)
    orig_layers = {l.name: l for l in original.layers}

    inp = tf.keras.Input(shape=(28, 28, 1), name="conv1_input")
    x   = tf.keras.layers.Conv2D(6,  5, activation="relu", name="conv1")(inp)
    x   = tf.keras.layers.AveragePooling2D(2,               name="pool1")(x)
    x   = tf.keras.layers.Conv2D(16, 5, activation="relu", name="conv2")(x)
    x   = tf.keras.layers.AveragePooling2D(2,               name="pool2")(x)
    x   = tf.keras.layers.Flatten(                          name="flatten")(x)
    x   = tf.keras.layers.Dense(120, activation="relu",     name="fc1")(x)
    x   = tf.keras.layers.Dense(84,  activation="relu",     name="fc2")(x)
    out = tf.keras.layers.Dense(10,                         name="output")(x)

    model = tf.keras.Model(inp, out)

    transferred, skipped = [], []
    for layer in model.layers:
        if layer.name in orig_layers:
            orig = orig_layers[layer.name]
            if orig.get_weights():
                try:
                    layer.set_weights(orig.get_weights())
                    transferred.append(layer.name)
                except ValueError as e:
                    skipped.append(f"{layer.name}: {e}")

    print(f"  ✅ Pesi trasferiti: {transferred}")
    if skipped:
        print(f"  ⚠️  Saltati: {skipped}")
    return model

# ─────────────────────────────────────────────────────────────────
# CONFIGURAZIONE HLS
# ─────────────────────────────────────────────────────────────────

def build_config(model):
    cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")
    cfg["Model"]["Precision"]   = PRECISION
    cfg["Model"]["ReuseFactor"] = RF["default"]
    cfg["Model"]["Strategy"]    = STRATEGY

    print(f"\n{'Layer':<15} {'MAC':>6}  {'RF':>6}  {'Parallelo':>10}  Assertion OK?")
    print("─" * 60)

    for layer in model.layers:
        name = layer.name
        if name not in cfg["LayerName"]:
            cfg["LayerName"][name] = {}

        is_pool = "pool" in layer.__class__.__name__.lower()
        rf      = RF["pool"] if is_pool else RF.get(name, RF["default"])
        mac     = MAC.get(name, None)

        cfg["LayerName"][name]["Precision"]   = PRECISION
        cfg["LayerName"][name]["ReuseFactor"] = rf
        cfg["LayerName"][name]["Strategy"]    = STRATEGY

        # Verifica divisione esatta
        if mac:
            ok  = "✅" if mac % rf == 0 else "⚠️  NON DIVIDE"
            par = mac // rf
            print(f"  {name:<13} {mac:>6}  {rf:>6}  {par:>10}  {ok}")
        else:
            print(f"  {name:<13} {'—':>6}  {rf:>6}  {'—':>10}")

    return cfg

# ─────────────────────────────────────────────────────────────────
# TESTBENCH — genera i .dat e li mette in OUTPUT_DIR/tb/
# ─────────────────────────────────────────────────────────────────

def generate_testbench(model, out_dir, n_samples=100):
    """
    Genera tb_input_features.dat e tb_output_predictions.dat
    dentro <out_dir>/tb/ così vengono inclusi nello zip.

    I float in input sono normalizzati [0,1] — nessuna pre-quantizzazione
    manuale: la quantizzazione ap_fixed<8,3> è responsabilità di hls4ml/Bambu.
    """
    tb_dir = os.path.join(out_dir, "tb")
    os.makedirs(tb_dir, exist_ok=True)

    (_, _), (x_test, y_test) = mnist.load_data()
    x = x_test[:n_samples].astype(np.float32) / 255.0
    x = x.reshape(-1, 28, 28, 1)

    preds = model.predict(x, verbose=0)

    np.savetxt(
        os.path.join(tb_dir, "tb_input_features.dat"),
        x.reshape(n_samples, -1),
        fmt="%.6f",
        delimiter=" ",
    )
    np.savetxt(
        os.path.join(tb_dir, "tb_output_predictions.dat"),
        preds,
        fmt="%.6f",
        delimiter=" ",
    )

    # Salva anche le label reali per confronto post-simulazione
    np.savetxt(
        os.path.join(tb_dir, "tb_true_labels.dat"),
        y_test[:n_samples].reshape(-1, 1),
        fmt="%d",
    )

    accuracy = np.mean(np.argmax(preds, axis=1) == y_test[:n_samples])
    print(f"  ✅ Testbench generato in '{tb_dir}/'")
    print(f"     Campioni: {n_samples}")
    print(f"     Input:    {x.reshape(n_samples,-1).shape}")
    print(f"     Output:   {preds.shape}")
    print(f"     Accuracy (float): {accuracy*100:.1f}%")
    print(f"     Esempio [0]: pred={np.argmax(preds[0])}, label={y_test[0]}")

# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def run():
    import sys

    if not os.path.isfile(MODEL_PATH):
        sys.exit(
            f"\nERRORE: '{MODEL_PATH}' non trovato.\n"
            "Carica il file .h5 prima di eseguire:\n"
            "  from google.colab import files; files.upload()\n"
        )

    # Pulizia run precedenti
    for path in [OUTPUT_DIR, f"{BUNDLE_ZIP}.zip"]:
        if os.path.exists(path):
            shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

    # ── Riepilogo RF pianificati ──────────────────────────────────
    print("═" * 60)
    print("RF pianificati:")
    for name, mac in MAC.items():
        rf = RF[name]
        lim_note = ""
        if "conv" in name:
            # Calcola il limite dell'assertion
            parts = name  # "conv1" o "conv2"
            lim = 5*5*1 if name == "conv1" else 5*5*6
            lim_note = f"  (limite assertion: {lim})"
        print(f"  {name:<10}  MAC={mac:<6}  RF={rf}{lim_note}")
    print("═" * 60)

    # ── Build modello + config ────────────────────────────────────
    print("\nCaricamento modello...")
    model = build_model()
    print()
    cfg = build_config(model)

    # ── Conversione HLS → C++ ─────────────────────────────────────
    print(f"\nConversione → C++ (Backend: Bambu, {1000//CLOCK_PERIOD_NS} MHz)...")
    hls_model = hls4ml.converters.convert_from_keras_model(
        model,
        hls_config=cfg,
        output_dir=OUTPUT_DIR,
        backend="Bambu",
        clock_period=CLOCK_PERIOD_NS,
    )
    hls_model.write()
    print(f"  ✅ File C++ scritti in '{OUTPUT_DIR}/'")

    # ── Testbench (dentro OUTPUT_DIR/tb/ → incluso nello zip) ─────
    print("\nGenerazione testbench...")
    generate_testbench(model, OUTPUT_DIR, n_samples=100)

    # ── Archivio finale: root_dir + base_dir per zip pulito ───────
    shutil.make_archive(
        BUNDLE_ZIP,          # nome output (senza .zip)
        "zip",
        root_dir=".",        # punto di partenza
        base_dir=OUTPUT_DIR  # cartella da comprimere (percorso relativo)
    )
    size_mb = os.path.getsize(f"{BUNDLE_ZIP}.zip") / 1e6
    print(f"\n  ✅ Archivio: {BUNDLE_ZIP}.zip ({size_mb:.1f} MB)")
    print(f"     Struttura zip:")
    print(f"       {OUTPUT_DIR}/")
    print(f"       {OUTPUT_DIR}/firmware/      ← C++ generato da hls4ml")
    print(f"       {OUTPUT_DIR}/tb/            ← testbench .dat")
    print(f"         tb_input_features.dat")
    print(f"         tb_output_predictions.dat")
    print(f"         tb_true_labels.dat")

    print("\n" + "═"*60)
    print("COMPLETATO — prossimi passi sulla VM:")
    print("  1. Estrai lo .zip")
    print("  2. Esegui fix_all.sh")
    print("  3. Bambu: --clock-period=40 --bambu-parameter=inline-max-cost=0")
    print("  4. Importa myproject.v + panda_libtech.v in Lattice Diamond")
    print("  5. Target: LFE5UM5G-85F-8BG381 → Map → P&R → Bitstream")
    print("═"*60)

    files.download(f"{BUNDLE_ZIP}.zip")

run()

════════════════════════════════════════════════════════════
RF pianificati:
  conv1       MAC=150     RF=25  (limite assertion: 25)
  conv2       MAC=2400    RF=150  (limite assertion: 150)
  fc1         MAC=48000   RF=128
  fc2         MAC=10080   RF=84
  output      MAC=840     RF=84
════════════════════════════════════════════════════════════

Caricamento modello...
  ✅ Pesi trasferiti: ['conv1', 'conv2', 'fc1', 'fc2', 'output']


Layer              MAC      RF   Parallelo  Assertion OK?
────────────────────────────────────────────────────────────
  conv1_input        —       8           —
  conv1            150      25           6  ✅
  pool1              —       1           —
  conv2           2400     150          16  ✅
  pool2              —       1           —
  flatten            —       8           —
  fc1            48000     128         375  ✅
  fc2            10080      84         120  ✅
  output           840      84          10  ✅

Conversione → C++ (Backend: Bambu, 25 M

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>